# OILTRACE AI — CNN Notebook (final)
**SAR oil-spill classifier · leakage-free evaluation · calibrated probability · explainable mask & geometry · deployable module**

This notebook is the satellite-image half of OILTRACE AI. It is **independent of the AIS notebook**.

| Output file | Used for |
|---|---|
| `oiltrace_cnn.py` | importable backend module: model, `OilSpillPredictor`, Grad-CAM mask, geometry, large-scene tiling |
| `oilspill_cnn_best.pth` | trained weights (plain `state_dict`) |
| `cnn_config.json` | preprocessing + decision threshold + temperature + provenance (the backend reads this) |
| `cnn_test_metrics.json`, `cnn_training_history.csv`, `cnn_split_manifest.csv` | reproducible, honest results |
| `cnn_output_example.json` + overlay PNGs | frozen result contract for `/detect-spill` and `/characterize-spill` |
| `OILTRACE_CNN_HANDOFF.zip` | everything above in one download |

### What this version fixes
1. **Data leakage.** The dataset's original file names show the 5,548 images are **overlapping 400×400 tiles (50 % overlap)** cut from larger scenes,
   plus rotated/cropped copies of the same scene. A *random* split puts overlapping pixels of one scene in both train and test, which inflates accuracy.
   Here the split is **grouped by scene** (`StratifiedGroupKFold`) and exact duplicates / cross-class duplicates are removed. Expect lower — but honest — numbers.
2. **Reproducible threshold.** The decision threshold is **selected on the validation set inside this notebook** (the old notebook hard-coded 0.51).
3. **Probability calibration.** Temperature scaling fitted on validation (minimises log-loss); ECE and a reliability plot are reported so you can *see* how trustworthy the percentage is.
4. **Confidence intervals** (scene-cluster bootstrap) and a **per-region** breakdown instead of one lucky number.
5. **Deployment gap closed:** single-image `predict`, test-time-augmentation uncertainty, Grad-CAM mask, geometry (area / length / width), large-scene tiling,
   a weights-reload consistency test and a frozen JSON contract.
6. Training and deployment share **one** model/preprocessing implementation (`oiltrace_cnn.py`).

### What this model is — and is not
- ✅ A binary patch classifier: *oil-like dark patch* vs *not*, with calibrated probability.
- ⚠️ **Not a segmentation network.** The mask is a Grad-CAM region refined by darkness (approximate). Geometry depends on the assumed pixel size.
- ⚠️ **Not a look-alike classifier.** It has two classes; it cannot split "not oil" into biofilm / low-wind / other. The module says so explicitly in its output.

> *Dark SAR patches are not automatically oil. Output is an investigation aid, not a legal determination.*

## 1. Setup and configuration

In [ ]:
# Colab: everything below is preinstalled. Elsewhere:  pip install torch torchvision pillow scipy scikit-learn pandas matplotlib
import os, sys, io, re, json, glob, time, copy, math, hashlib, random, shutil, subprocess, zipfile, importlib
from datetime import datetime, timezone
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision, sklearn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score

SEED = 42
SMOKE_TEST = os.environ.get("OILTRACE_SMOKE") == "1"   # tiny run to verify the pipeline only (leave False for real training)
FAST_CPU = os.environ.get("OILTRACE_FAST") == "1"      # full data, 64-px, 6 epochs: indicative CPU run (NOT the final model)

CFG = dict(
    image_size=64 if (SMOKE_TEST or FAST_CPU) else 224,
    batch_size=32,
    max_epochs=2 if SMOKE_TEST else 6 if FAST_CPU else 50,
    patience=4 if FAST_CPU else 8,                  # early stopping on validation ROC-AUC
    lr=1e-3, weight_decay=1e-4, dropout=0.30,
    threshold_mode="max_f1",     # "max_f1"  or  "recall_floor" (highest precision with recall >= recall_floor)
    recall_floor=0.90,
    n_boot=100 if SMOKE_TEST else 300 if FAST_CPU else 1000,
    smoke_max_images=360,
)
OUT_DIR = "OILTRACE_CNN_models"
os.makedirs(OUT_DIR, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0 if (SMOKE_TEST or DEVICE.type == "cpu") else 2

def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
seed_everything()
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device {DEVICE} | SMOKE_TEST={SMOKE_TEST} | FAST_CPU={FAST_CPU}")
if SMOKE_TEST or FAST_CPU:
    print("⚠️  Reduced-size run: do NOT ship these weights. Re-run with both flags off (use a GPU, e.g. Colab T4).")
print(CFG)

## 2. The deployable module (`oiltrace_cnn.py`)
Training below imports the model and transforms from this file — the backend will run **identical** code.

In [ ]:
%%writefile oiltrace_cnn.py
"""
OILTRACE AI - CNN module
========================
Single source of truth for the SAR oil-spill CNN: the SAME model class and preprocessing are used
for training (notebook) and for deployment (backend), so they cannot drift apart.

What this model IS
    A binary image-patch classifier (oil-like dark patch vs. not) trained on 400x400 Sentinel-1 tiles.
What it is NOT
    * It is NOT a trained segmentation network. The "mask" produced here is a weakly-supervised
      Grad-CAM region refined by darkness - an APPROXIMATION for visualisation and rough geometry.
    * It cannot separate oil from specific look-alikes (biofilm, low-wind area ...): it only has two
      classes. Look-alike percentages must be supplied by another component and labelled accordingly.
    * Probabilities are only meaningful for imagery resembling the training data.
"""
from __future__ import annotations

import base64
import io
import json
import os
from typing import Dict, List, Optional, Sequence, Tuple, Union

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFilter
from torchvision import transforms

__all__ = ["ConvBlock", "OilSpillCNN", "build_transforms", "OilSpillPredictor", "CLASS_NAMES", "DISCLAIMER"]

CLASS_NAMES = {0: "NO OIL SPILL", 1: "OIL SPILL"}
DISCLAIMER = ("Image-patch classifier trained on Sentinel-1 tiles. Dark SAR features are not automatically oil; "
              "mask and geometry are approximate (Grad-CAM derived), not a trained segmentation.")


# --------------------------------------------------------------------------
# Model
# --------------------------------------------------------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)


class OilSpillCNN(nn.Module):
    """4 conv blocks (2x Conv-BN-ReLU each) + global average pooling + small MLP head."""

    def __init__(self, dropout: float = 0.30, num_classes: int = 2):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3, 32), ConvBlock(32, 64), ConvBlock(64, 128), ConvBlock(128, 256),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(dropout), nn.Linear(256, 128), nn.ReLU(inplace=True),
            nn.Dropout(dropout), nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def build_transforms(image_size: int = 224, mean: float = 0.5, std: float = 0.5, train: bool = False):
    """Grayscale SAR -> 3 identical channels -> [-1, 1]. `train=True` adds mild geometric augmentation."""
    aug = ([transforms.RandomHorizontalFlip(0.5), transforms.RandomVerticalFlip(0.5),
            transforms.RandomRotation(15)] if train else [])
    return transforms.Compose([
        transforms.Resize((image_size, image_size)), *aug,
        transforms.Grayscale(num_output_channels=3), transforms.ToTensor(),
        transforms.Normalize([mean] * 3, [std] * 3),
    ])


# --------------------------------------------------------------------------
# helpers
# --------------------------------------------------------------------------
ImageLike = Union[str, bytes, np.ndarray, Image.Image]


def _to_gray_pil(img: ImageLike) -> Image.Image:
    if isinstance(img, Image.Image):
        return img.convert("L")
    if isinstance(img, (bytes, bytearray)):
        return Image.open(io.BytesIO(img)).convert("L")
    if isinstance(img, np.ndarray):
        a = img
        if a.ndim == 3:
            a = a[..., :3].astype(np.float64).mean(axis=2)
        if a.dtype != np.uint8:
            # float / 16-bit rasters (e.g. dB backscatter): NaN-safe 1-99 percentile stretch to 8 bit.
            # NOTE: the training JPEGs' exact stretch is unknown - verify on your own scenes.
            a = a.astype(np.float64)
            finite = np.isfinite(a)
            if not finite.any():
                a = np.zeros(a.shape, np.uint8)
            else:
                lo, hi = np.percentile(a[finite], [1, 99])
                a = np.nan_to_num(a, nan=lo, posinf=hi, neginf=lo)
                a = (np.clip((a - lo) / (hi - lo + 1e-9), 0, 1) * 255).astype(np.uint8)
        return Image.fromarray(a).convert("L")
    with Image.open(img) as im:
        return im.convert("L")


def _otsu(values: np.ndarray) -> float:
    hist, edges = np.histogram(values, bins=64)
    p = hist / max(hist.sum(), 1)
    centers = (edges[:-1] + edges[1:]) / 2
    w0, w1 = np.cumsum(p), 1 - np.cumsum(p)
    m0 = np.cumsum(p * centers) / np.maximum(w0, 1e-9)
    mt = (p * centers).sum()
    m1 = (mt - np.cumsum(p * centers)) / np.maximum(w1, 1e-9)
    return float(centers[np.argmax(w0 * w1 * (m0 - m1) ** 2)])


def _geometry(mask: np.ndarray, pixel_size_m: float) -> Dict:
    from scipy import ndimage as ndi
    n = int(mask.sum())
    px_km2 = (pixel_size_m / 1000.0) ** 2
    out = {"area_km2": round(n * px_km2, 4), "pixel_count": n, "coverage_fraction": round(n / mask.size, 4),
           "length_km": None, "width_km": None, "orientation_deg": None, "n_components": 0}
    if n < 50:
        return out
    lab, k = ndi.label(mask)
    out["n_components"] = int(k)
    sizes = ndi.sum(mask, lab, index=range(1, k + 1))
    big = lab == (int(np.argmax(sizes)) + 1)
    ys, xs = np.nonzero(big)
    pts = np.stack([xs, -ys], 1).astype(float)
    pts -= pts.mean(0)
    evals, evecs = np.linalg.eigh(np.cov(pts.T))
    major, minor = evecs[:, 1], evecs[:, 0]
    pm, pn = pts @ major, pts @ minor
    out["length_km"] = round(float((pm.max() - pm.min() + 1) * pixel_size_m / 1000), 3)
    out["width_km"] = round(float((pn.max() - pn.min() + 1) * pixel_size_m / 1000), 3)
    out["orientation_deg"] = round(float((np.degrees(np.arctan2(major[0], major[1])) + 360) % 180), 1)
    return out


# --------------------------------------------------------------------------
# Predictor
# --------------------------------------------------------------------------
class OilSpillPredictor:
    """
    Deployment wrapper. Loads weights + cnn_config.json produced by the training notebook.

        pred = OilSpillPredictor.load("OILTRACE_models")
        pred.predict("tile.jpg")                 # patch-level result (with test-time augmentation)
        pred.analyze(big_scene_array, pixel_size_m=10)   # tiles a large scene, mask + geometry
    """

    def __init__(self, model: nn.Module, config: Dict, device: Optional[str] = None):
        self.cfg = config
        self.device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.model = model.to(self.device).eval()
        self.size = int(config["image_size"])
        self.threshold = float(config["decision_threshold"])
        self.temperature = float(config.get("temperature", 1.0))
        self.tf = build_transforms(self.size, config["normalize_mean"], config["normalize_std"], train=False)
        self.tile = int(config.get("native_tile_px", 400))

    # ---- construction
    @classmethod
    def load(cls, model_dir: str, device: Optional[str] = None) -> "OilSpillPredictor":
        with open(os.path.join(model_dir, "cnn_config.json"), encoding="utf-8") as f:
            cfg = json.load(f)
        model = OilSpillCNN(dropout=cfg.get("dropout", 0.30))
        state = torch.load(os.path.join(model_dir, cfg.get("weights_file", "oilspill_cnn_best.pth")),
                           map_location="cpu")
        model.load_state_dict(state)
        return cls(model, cfg, device)

    # ---- core inference
    @torch.no_grad()
    def _logits(self, batch: torch.Tensor) -> torch.Tensor:
        return self.model(batch.to(self.device))

    def _prob_from_logits(self, logits: torch.Tensor) -> torch.Tensor:
        return torch.softmax(logits / self.temperature, dim=1)[:, 1]

    @torch.no_grad()
    def predict_tile(self, img: ImageLike, tta: bool = True) -> Dict:
        """Classify ONE tile. With tta=True the probability is averaged over 4 flips (std = uncertainty)."""
        pil = _to_gray_pil(img)
        flat = float(np.asarray(pil, dtype=np.float32).std()) < 2.0
        x = self.tf(pil).unsqueeze(0)
        views = [x] + ([x.flip(-1), x.flip(-2), x.flip(-1).flip(-2)] if tta else [])
        probs = torch.cat([self._prob_from_logits(self._logits(v)) for v in views]).cpu().numpy()
        p, sd = float(probs.mean()), float(probs.std())
        out = self._package(p, sd)
        out["warnings"] = ["Image is almost uniform (blank / no-data?) - result is unreliable"] if flat else []
        return out

    def _package(self, p: float, sd: float) -> Dict:
        label = int(p >= self.threshold)
        decisiveness = max(p, 1 - p)
        conf = ("High" if decisiveness >= 0.85 and sd <= 0.10 else
                "Medium" if decisiveness >= 0.65 and sd <= 0.20 else "Low")
        return {
            "oil_probability": round(p, 4), "prediction": CLASS_NAMES[label], "class": label,
            "confidence": conf, "tta_std": round(sd, 4), "decision_threshold": self.threshold,
            "lookalike_analysis": {
                "oil_like": round(p, 4), "not_oil_like": round(1 - p, 4), "breakdown_available": False,
                "note": "Binary model: cannot split the non-oil share into biofilm / low-wind / other. "
                        "Any such split shown in the UI must come from another component and be labelled demo.",
            },
        }

    def predict(self, img: ImageLike, tta: bool = True) -> Dict:
        """Alias of predict_tile for a single tile/patch."""
        return self.predict_tile(img, tta=tta)

    def predict_batch(self, images: Sequence[ImageLike], batch_size: int = 32) -> np.ndarray:
        """Oil probabilities (no TTA) for many tiles - used for evaluation/consistency checks."""
        out = []
        for i in range(0, len(images), batch_size):
            xb = torch.stack([self.tf(_to_gray_pil(im)) for im in images[i:i + batch_size]])
            out.append(self._prob_from_logits(self._logits(xb)).cpu().numpy())
        return np.concatenate(out) if out else np.array([])

    # ---- explainability
    def gradcam(self, img: ImageLike) -> np.ndarray:
        """Grad-CAM for the 'oil' class on one tile, resized to the tile, values in [0, 1]."""
        pil = _to_gray_pil(img)
        x = self.tf(pil).unsqueeze(0).to(self.device)
        store = {}
        h = self.model.features[3].register_forward_hook(lambda m, i, o: store.__setitem__("a", o))
        try:
            with torch.enable_grad():
                logits = self.model(x)
                grads = torch.autograd.grad(logits[0, 1], store["a"])[0]
        finally:
            h.remove()
        a = store["a"].detach()
        w = grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * a).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=(pil.height, pil.width), mode="bilinear", align_corners=False)[0, 0]
        cam = cam.cpu().numpy()
        return cam / cam.max() if cam.max() > 0 else cam

    def tile_mask(self, img: ImageLike, cam_threshold: float = 0.5, refine_dark: bool = True) -> Tuple[np.ndarray, np.ndarray]:
        """Approximate slick mask for ONE tile: Grad-CAM >= threshold, optionally restricted to dark pixels."""
        pil = _to_gray_pil(img)
        cam = self.gradcam(pil)
        mask = cam >= cam_threshold
        if refine_dark and mask.sum() > 100:
            g = np.asarray(pil.filter(ImageFilter.GaussianBlur(2)), dtype=float)
            t = _otsu(g[mask])                    # dark/bright split inside the CAM region
            mask = mask & (g <= t)
            from scipy import ndimage as ndi
            mask = ndi.binary_opening(mask, iterations=1)
            mask = ndi.binary_closing(mask, iterations=2)
        return mask, cam

    # ---- scene level
    def analyze(self, img: ImageLike, pixel_size_m: float = 10.0, stride: Optional[int] = None,
                max_tiles: int = 400) -> Dict:
        """
        Full analysis of a tile or a LARGE scene.
        Tiles the image (native tile size, 50 % overlap), classifies every tile, builds a probability
        map, and for tiles above threshold derives the approximate mask; then computes geometry.

        pixel_size_m is an ASSUMPTION unless you know the scene's ground sampling distance
        (Sentinel-1 GRD IW is ~10 m). Area/length/width scale with it.
        """
        pil = _to_gray_pil(img)
        W, H = pil.size
        T = self.tile
        stride = stride or T // 2
        mask = np.zeros((H, W), bool)
        prob_map = np.zeros((H, W), np.float32)
        count = np.zeros((H, W), np.float32)
        tile_results = []

        if W <= T * 1.25 and H <= T * 1.25:
            boxes = [(0, 0, W, H)]
        else:
            xs = list(range(0, max(W - T, 0) + 1, stride))
            ys = list(range(0, max(H - T, 0) + 1, stride))
            if xs[-1] + T < W:
                xs.append(W - T)
            if ys[-1] + T < H:
                ys.append(H - T)
            boxes = [(x, y, min(x + T, W), min(y + T, H)) for y in ys for x in xs]
            if len(boxes) > max_tiles:
                raise ValueError(f"{len(boxes)} tiles exceed max_tiles={max_tiles}; increase stride/max_tiles")

        for (x0, y0, x1, y1) in boxes:
            crop = pil.crop((x0, y0, x1, y1))
            r = self.predict_tile(crop, tta=True)
            tile_results.append({"box": [x0, y0, x1, y1], "oil_probability": r["oil_probability"],
                                 "tta_std": r["tta_std"], "warnings": r["warnings"]})
            prob_map[y0:y1, x0:x1] += r["oil_probability"]
            count[y0:y1, x0:x1] += 1
            if r["class"] == 1:
                m, _ = self.tile_mask(crop)
                mask[y0:y1, x0:x1] |= m
        prob_map /= np.maximum(count, 1)

        probs = np.array([t["oil_probability"] for t in tile_results])
        best = tile_results[int(probs.argmax())]
        scene = self._package(float(probs.max()), float(best["tta_std"]))
        scene["warnings"] = sorted({w for t in tile_results for w in t["warnings"]})
        scene["scene_decision_rule"] = ("scene P(oil) = highest tile probability; in very large scenes this rule "
                                        "raises the false-alarm rate - check n_positive_tiles")
        scene.update({
            "n_tiles": len(boxes), "n_positive_tiles": int((probs >= self.threshold).sum()),
            "best_tile": best, "image_size_px": [W, H],
            "geometry": _geometry(mask, pixel_size_m), "pixel_size_m": pixel_size_m,
            "geometry_assumption": "Approximate (Grad-CAM-derived mask). pixel_size_m must match the scene.",
            "disclaimer": DISCLAIMER,
        })
        return {"result": scene, "mask": mask, "prob_map": prob_map, "gray": np.asarray(pil)}

    # ---- rendering for the UI
    @staticmethod
    def render_overlay(gray: np.ndarray, mask: np.ndarray, alpha: float = 0.45) -> Image.Image:
        base = np.stack([gray] * 3, -1).astype(float)
        col = np.array([255, 110, 30], float)
        out = base.copy()
        out[mask] = (1 - alpha) * base[mask] + alpha * col
        return Image.fromarray(out.astype(np.uint8))

    @staticmethod
    def to_base64_png(img: Union[Image.Image, np.ndarray]) -> str:
        if isinstance(img, np.ndarray):
            img = Image.fromarray(img.astype(np.uint8) * (255 if img.dtype == bool else 1))
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        return base64.b64encode(buf.getvalue()).decode("ascii")

In [ ]:
import oiltrace_cnn as C
importlib.reload(C)
m_ = C.OilSpillCNN(dropout=CFG["dropout"])
print("parameters:", f"{sum(p.numel() for p in m_.parameters()):,}", "| class names:", C.CLASS_NAMES)

## 3. Data
Source: the public repository used by the original project (it keeps the **original file names**, which carry the scene id needed for a leakage-free split).
A Kaggle copy renames files to `class_N_xxxxx.jpg` and **loses** the scene id — do not use it for the split.

In [ ]:
REPO_URL = "https://github.com/Saisamarth21/Oil-Spill-Detection-in-Marine-Environments-Using-AIS-and-Satellite-Data.git"
REPO_DIR = "_oiltrace_repo"
DATA_DIR = None        # <- optionally set to a folder that contains sub-folders '0' and '1' (original file names)

def locate_data():
    cands = [DATA_DIR] + [f"{REPO_DIR}/SAR Image Dataset"]
    for c in cands:
        if c and os.path.isdir(os.path.join(c, "0")) and os.path.isdir(os.path.join(c, "1")):
            return c
    print("Cloning the public repository (needs internet; ~160 MB) ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    return f"{REPO_DIR}/SAR Image Dataset"

DATA_DIR = locate_data()
print("data folder:", DATA_DIR)

In [ ]:
# Parse original file names:  <tile>_<x>_<y>_img_<SCENE>_<REGION>_...[augmentation]..._cls_<label>.jpg
pat = re.compile(r"^(\d+)_(\d+)_(\d+)_img_([A-Za-z0-9]+)_([A-Za-z0-9]+)_")
rows, unparsed = [], 0
for lab in ("0", "1"):
    for f in sorted(os.listdir(os.path.join(DATA_DIR, lab))):
        if not f.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        m = pat.match(f)
        if m:
            tile, x, y, scene, region = m.group(1), int(m.group(2)), int(m.group(3)), m.group(4), m.group(5)
        else:
            tile, x, y, scene, region = "?", -1, -1, "u_" + hashlib.md5(f.encode()).hexdigest()[:10], "UNK"; unparsed += 1
        rows.append(dict(path=os.path.join(DATA_DIR, lab, f), file=f, label=int(lab), scene=scene, region=region, x=x, y=y,
                         augmented=bool(re.search(r"(rotate|slant|crop|flip|shear)\d*", f))))
df = pd.DataFrame(rows)
print(f"images: {len(df)} | class 0: {(df.label==0).sum()} | class 1: {(df.label==1).sum()} | unparsed names: {unparsed}")
print(f"unique scenes: {df.scene.nunique()} | tiles per scene: mean {len(df)/df.scene.nunique():.2f}, max {df.groupby('scene').size().max()}")
print(f"tile offsets seen (px): {[int(v) for v in sorted(df.loc[df.x>=0,'x'].unique())[:8]]}  -> stride 200 on 400-px tiles = 50 % overlap")
print(f"augmented copies (rotate/slant/crop): {int(df.augmented.sum())}")
print("regions:", df.region.value_counts().head(8).to_dict())

In [ ]:
# Exact duplicates (same bytes) and duplicates that carry BOTH labels (label noise) are removed.
df["md5"] = [hashlib.md5(open(p, "rb").read()).hexdigest() for p in df.path]
lab_per_hash = df.groupby("md5").label.nunique()
conflict = set(lab_per_hash[lab_per_hash > 1].index)
n_conf = int(df.md5.isin(conflict).sum())
df = df[~df.md5.isin(conflict)]
n_dup = int(df.duplicated("md5").sum())
df = df.drop_duplicates("md5").reset_index(drop=True)
print(f"removed {n_conf} images whose identical copy has the opposite label, and {n_dup} exact duplicates -> {len(df)} images remain")

if SMOKE_TEST:       # tiny stratified subset, whole scenes kept together
    sc = df.groupby("scene").label.max().reset_index().sample(frac=1, random_state=SEED)
    keep, cnt = [], 0
    for s in sc.scene:
        n = int((df.scene == s).sum())
        if cnt + n > CFG["smoke_max_images"]:
            continue
        keep.append(s); cnt += n
    df = df[df.scene.isin(keep)].reset_index(drop=True)
    print("SMOKE_TEST subset:", len(df), "images,", df.label.value_counts().to_dict())

## 4. Leakage-free split (grouped by scene)
First, *measure* the leakage a random split would have caused, then split properly: **80 % train / 10 % val / 10 % test**, with every scene in exactly one split.

In [ ]:
# Diagnostic: how many test tiles would share a scene with training tiles under a naive random split?
rs = np.random.RandomState(SEED).permutation(len(df))
n_te = int(0.1 * len(df)); naive_test, naive_train = rs[:n_te], rs[n_te:]
shared = df.iloc[naive_test].scene.isin(set(df.iloc[naive_train].scene))
print(f"NAIVE random split: {shared.mean():.1%} of test tiles come from a scene that is also in the training set  (this is the leakage)")

# Proper split
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
folds = [te for _, te in sgkf.split(df, df.label, groups=df.scene)]
df["split"] = "train"
df.loc[df.index[folds[0]], "split"] = "test"
df.loc[df.index[folds[1]], "split"] = "val"
S = {k: set(df[df.split == k].scene) for k in ("train", "val", "test")}
assert not (S["train"] & S["val"]) and not (S["train"] & S["test"]) and not (S["val"] & S["test"]), "scene leakage between splits!"
print("GROUPED split: no scene appears in more than one split (asserted)\n")
print(pd.crosstab(df.split, df.label, margins=True).rename(columns={0: "no oil", 1: "oil"}))
train_df, val_df, test_df = (df[df.split == k].reset_index(drop=True) for k in ("train", "val", "test"))

## 5. Datasets, model, training

In [ ]:
class SARDataset(Dataset):
    def __init__(self, frame, tf):
        self.paths, self.labels, self.tf = frame.path.tolist(), frame.label.tolist(), tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            img = im.convert("L")
        return self.tf(img), self.labels[i]

tf_train = C.build_transforms(CFG["image_size"], 0.5, 0.5, train=True)
tf_eval = C.build_transforms(CFG["image_size"], 0.5, 0.5, train=False)
mk = lambda d, tf, sh: DataLoader(SARDataset(d, tf), batch_size=CFG["batch_size"], shuffle=sh, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda")
train_loader, val_loader, test_loader = mk(train_df, tf_train, True), mk(val_df, tf_eval, False), mk(test_df, tf_eval, False)
print(len(train_df), len(val_df), len(test_df), "images")

In [ ]:
@torch.no_grad()
def get_logits(model, loader):
    model.eval(); out, ys = [], []
    for x, y in loader:
        out.append(model(x.to(DEVICE)).float().cpu()); ys.append(y)
    return torch.cat(out), torch.cat(ys)

def metrics(y, p, thr=0.5):
    y, p = np.asarray(y), np.asarray(p); pred = (p >= thr).astype(int)
    tp = int(((pred == 1) & (y == 1)).sum()); tn = int(((pred == 0) & (y == 0)).sum())
    fp = int(((pred == 1) & (y == 0)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    return dict(accuracy=(tp + tn) / max(len(y), 1), precision=prec, recall=rec,
                f1=2 * prec * rec / max(prec + rec, 1e-9), specificity=tn / max(tn + fp, 1),
                roc_auc=float(roc_auc_score(y, p)) if len(set(y.tolist())) > 1 else float("nan"),
                tn=tn, fp=fp, fn=fn, tp=tp)

model = C.OilSpillCNN(dropout=CFG["dropout"]).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

history, best_auc, best_state, best_epoch, bad = [], -1.0, None, 0, 0
t0 = time.time()
for epoch in range(1, CFG["max_epochs"] + 1):
    model.train(); tl, n = 0.0, 0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == "cuda"):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        scaler.step(optimizer); scaler.update()
        tl += loss.item() * len(y); n += len(y)
    lv, yv = get_logits(model, val_loader)
    pv = torch.softmax(lv, 1)[:, 1].numpy(); mv = metrics(yv.numpy(), pv)
    vloss = float(criterion(lv, yv))
    scheduler.step(mv["roc_auc"])
    history.append(dict(epoch=epoch, train_loss=tl / n, val_loss=vloss, val_acc=mv["accuracy"], val_f1=mv["f1"],
                        val_auc=mv["roc_auc"], lr=optimizer.param_groups[0]["lr"]))
    flag = ""
    if mv["roc_auc"] > best_auc:
        best_auc, best_epoch, bad = mv["roc_auc"], epoch, 0
        best_state = copy.deepcopy(model.state_dict()); flag = "  <- best"
    else:
        bad += 1
    print(f"epoch {epoch:02d} | train loss {tl/n:.4f} | val loss {vloss:.4f} acc {mv['accuracy']:.3f} f1 {mv['f1']:.3f} auc {mv['roc_auc']:.3f} | {time.time()-t0:.0f}s{flag}")
    if bad >= CFG["patience"]:
        print(f"early stopping (no val-AUC gain for {CFG['patience']} epochs)"); break
model.load_state_dict(best_state)
print(f"\nbest epoch {best_epoch} (val ROC-AUC {best_auc:.4f})")
hist = pd.DataFrame(history)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(hist.epoch, hist.train_loss, label="train"); ax[0].plot(hist.epoch, hist.val_loss, label="val"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(hist.epoch, hist.val_auc, label="val AUC"); ax[1].plot(hist.epoch, hist.val_f1, label="val F1"); ax[1].set_title("Validation"); ax[1].legend()
for a in ax: a.set_xlabel("epoch"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6. Calibration and decision threshold — both fitted on the **validation** set only
Temperature scaling minimises validation log-loss (it does not guarantee a lower ECE — both are printed, check the reliability plot). The threshold is then chosen from an explicit sweep (no hard-coded value).

In [ ]:
def ece(y, p, bins=10):
    y, p = np.asarray(y), np.asarray(p); edges = np.linspace(0, 1, bins + 1); tot = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p > lo) & (p <= hi) if lo > 0 else (p >= lo) & (p <= hi)
        if m.any(): tot += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(tot)

def fit_temperature(logits, y):
    T = nn.Parameter(torch.ones(1)); opt = torch.optim.LBFGS([T], lr=0.1, max_iter=200)
    ce = nn.CrossEntropyLoss()
    def closure():
        opt.zero_grad(); loss = ce(logits / T.clamp(min=0.05), y); loss.backward(); return loss
    opt.step(closure)
    return float(T.clamp(min=0.05).item())

lv, yv = get_logits(model, val_loader)
TEMPERATURE = fit_temperature(lv, yv)
p_raw = torch.softmax(lv, 1)[:, 1].numpy(); p_cal = torch.softmax(lv / TEMPERATURE, 1)[:, 1].numpy()
print(f"temperature T = {TEMPERATURE:.3f}  (T>1: raw model was over-confident)")
print(f"validation ECE  raw {ece(yv.numpy(), p_raw):.4f}  ->  calibrated {ece(yv.numpy(), p_cal):.4f}")

ths = np.round(np.arange(0.05, 0.951, 0.01), 2)
sweep = pd.DataFrame([{"threshold": t, **{k: v for k, v in metrics(yv.numpy(), p_cal, t).items() if k in ("accuracy", "precision", "recall", "f1")}} for t in ths])
if CFG["threshold_mode"] == "recall_floor":
    ok = sweep[sweep.recall >= CFG["recall_floor"]]
    THRESHOLD = float(ok.sort_values(["precision", "threshold"], ascending=[False, False]).iloc[0].threshold) if len(ok) else 0.5
else:
    best = sweep.f1.max(); cand = sweep[sweep.f1 >= best - 1e-9]
    THRESHOLD = float(cand.iloc[(cand.threshold - 0.5).abs().argmin()].threshold)   # ties -> closest to 0.5
print(f"decision threshold ({CFG['threshold_mode']}) selected on VALIDATION = {THRESHOLD:.2f}")
display(sweep[sweep.threshold.isin([0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, THRESHOLD])].drop_duplicates("threshold").round(3))

## 7. Final evaluation on the held-out **test** set (used once)
Test scenes were never used for training, early stopping, calibration or threshold selection.

In [ ]:
lt, yt = get_logits(model, test_loader)
pt = torch.softmax(lt / TEMPERATURE, 1)[:, 1].numpy(); yt_np = yt.numpy()
M = metrics(yt_np, pt, THRESHOLD)
print(f"TEST ({len(yt_np)} tiles from {test_df.scene.nunique()} unseen scenes) @ threshold {THRESHOLD:.2f}")
for k in ("accuracy", "precision", "recall", "f1", "specificity", "roc_auc"):
    print(f"  {k:12s}: {M[k]:.4f}")
print(f"  confusion   : TN {M['tn']}  FP {M['fp']}  FN {M['fn']}  TP {M['tp']}")
print(f"  average precision (PR-AUC): {average_precision_score(yt_np, pt):.4f} | test ECE: {ece(yt_np, pt):.4f}")

# scene-cluster bootstrap: resample WHOLE scenes (tiles of one scene are not independent)
groups = test_df.scene.values; uniq = np.unique(groups); idx_by = {g: np.where(groups == g)[0] for g in uniq}
rng = np.random.default_rng(SEED); boots = []
for _ in range(CFG["n_boot"]):
    pick = np.concatenate([idx_by[g] for g in rng.choice(uniq, len(uniq))])
    if len(set(yt_np[pick].tolist())) < 2: continue
    b = metrics(yt_np[pick], pt[pick], THRESHOLD); boots.append([b["accuracy"], b["precision"], b["recall"], b["roc_auc"]])
boots = np.array(boots); lo, hi = np.percentile(boots, 2.5, 0), np.percentile(boots, 97.5, 0)
CI = {n: [float(a), float(b)] for n, a, b in zip(["accuracy", "precision", "recall", "roc_auc"], lo, hi)}
print("\n95 % CI (scene-cluster bootstrap):", {k: [round(v[0], 3), round(v[1], 3)] for k, v in CI.items()})

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(15, 3.4))
cm = np.array([[M["tn"], M["fp"]], [M["fn"], M["tp"]]])
ax[0].imshow(cm, cmap="Blues"); ax[0].set_title("Confusion matrix (test)")
for i in range(2):
    for j in range(2): ax[0].text(j, i, cm[i, j], ha="center", va="center", fontsize=13)
ax[0].set_xticks([0, 1], ["no oil", "oil"]); ax[0].set_yticks([0, 1], ["no oil", "oil"]); ax[0].set_xlabel("predicted"); ax[0].set_ylabel("true")
fpr, tpr, _ = roc_curve(yt_np, pt); ax[1].plot(fpr, tpr); ax[1].plot([0, 1], [0, 1], "k:"); ax[1].set_title(f"ROC (AUC {M['roc_auc']:.3f})"); ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR")
pr, rc, _ = precision_recall_curve(yt_np, pt); ax[2].plot(rc, pr); ax[2].set_title("Precision-Recall"); ax[2].set_xlabel("recall"); ax[2].set_ylabel("precision")
edges = np.linspace(0, 1, 11); mids, acc_ = [], []
for lo_, hi_ in zip(edges[:-1], edges[1:]):
    m = (pt > lo_) & (pt <= hi_)
    if m.sum() >= 3: mids.append(pt[m].mean()); acc_.append(yt_np[m].mean())
ax[3].plot([0, 1], [0, 1], "k:"); ax[3].plot(mids, acc_, "o-"); ax[3].set_title("Reliability (calibrated)"); ax[3].set_xlabel("predicted P(oil)"); ax[3].set_ylabel("observed")
plt.tight_layout(); plt.show()

# Per-region robustness (regions with enough test tiles)
tr = test_df.assign(p=pt, pred=(pt >= THRESHOLD).astype(int))
tr["tp_"] = ((tr.pred == 1) & (tr.label == 1)).astype(int); tr["ok_"] = (tr.pred == tr.label).astype(int)
reg = tr.groupby("region").agg(tiles=("label", "size"), oil_share=("label", "mean"), accuracy=("ok_", "mean"), tp=("tp_", "sum"), pos=("label", "sum"))
reg["recall"] = reg.tp / reg.pos.clip(lower=1); reg = reg[["tiles", "oil_share", "accuracy", "recall"]]
display(reg[reg.tiles >= 10].sort_values("tiles", ascending=False).round(3))

In [ ]:
# Error gallery - look at what the model gets wrong
errs = tr[tr.pred != tr.label].copy(); errs["conf"] = (errs.p - 0.5).abs()
fp_ = errs[errs.label == 0].sort_values("conf", ascending=False).head(5); fn_ = errs[errs.label == 1].sort_values("conf", ascending=False).head(5)
fig, axs = plt.subplots(2, 5, figsize=(13, 5.4))
for row, (name, frame) in enumerate([("FALSE POSITIVE (no oil)", fp_), ("FALSE NEGATIVE (oil)", fn_)]):
    for j in range(5):
        a = axs[row, j]; a.axis("off")
        if j < len(frame):
            r = frame.iloc[j]; a.imshow(Image.open(r.path).convert("L"), cmap="gray"); a.set_title(f"{name}\nP(oil)={r.p:.2f} [{r.region}]", fontsize=7)
plt.tight_layout(); plt.show()

## 8. Save artifacts

In [ ]:
WEIGHTS_FILE = "oilspill_cnn_best.pth"
torch.save({k: v.cpu() for k, v in model.state_dict().items()}, os.path.join(OUT_DIR, WEIGHTS_FILE))
config = {
    "model": "OilSpillCNN", "weights_file": WEIGHTS_FILE, "image_size": CFG["image_size"], "native_tile_px": 400,
    "normalize_mean": 0.5, "normalize_std": 0.5, "input": "grayscale replicated to 3 channels",
    "dropout": CFG["dropout"], "temperature": TEMPERATURE, "decision_threshold": THRESHOLD,
    "threshold_mode": CFG["threshold_mode"], "class_map": {"0": "NO OIL SPILL", "1": "OIL SPILL"},
    "positive_class": 1, "best_epoch": best_epoch,
    "evaluation": {"split": "scene-grouped (StratifiedGroupKFold), 80/10/10", "n_test_tiles": int(len(test_df)),
                   "n_test_scenes": int(test_df.scene.nunique()), "smoke_test": SMOKE_TEST, "fast_cpu_run": FAST_CPU},
    "provenance": {"data": "Sentinel-1 SAR 400x400 tiles (public repo Saisamarth21/Oil-Spill-Detection-... 'SAR Image Dataset')",
                   "label_meaning": "folder 1 = oil spill, folder 0 = no oil (from the dataset's '_cls_N' naming; confirm with the data owner)",
                   "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                   "torch": torch.__version__, "torchvision": torchvision.__version__, "seed": SEED},
    "limitations": ["binary classifier, not segmentation", "mask/geometry derived from Grad-CAM (approximate)",
                    "no look-alike breakdown", "trained on one dataset; unseen sensors/regions may behave differently"],
}
json.dump(config, open(os.path.join(OUT_DIR, "cnn_config.json"), "w"), indent=2)
test_metrics = {"threshold": THRESHOLD, "temperature": TEMPERATURE, **{k: (float(v) if isinstance(v, (float, np.floating)) else v) for k, v in M.items()},
                "ci95_scene_bootstrap": CI, "ece_test": ece(yt_np, pt), "average_precision": float(average_precision_score(yt_np, pt)),
                "split": "scene-grouped", "n_test": int(len(yt_np)), "smoke_test": SMOKE_TEST, "fast_cpu_run": FAST_CPU}
json.dump(test_metrics, open(os.path.join(OUT_DIR, "cnn_test_metrics.json"), "w"), indent=2)
hist.to_csv(os.path.join(OUT_DIR, "cnn_training_history.csv"), index=False)
df[["file", "label", "scene", "region", "split", "md5"]].to_csv(os.path.join(OUT_DIR, "cnn_split_manifest.csv"), index=False)
print("saved to", OUT_DIR, "->", sorted(os.listdir(OUT_DIR)))

## 9. Load through the **deployment module** and prove it matches the notebook

In [ ]:
pred = C.OilSpillPredictor.load(OUT_DIR)
sub = test_df.head(48)
p_mod = pred.predict_batch(sub.path.tolist())
xs = torch.stack([tf_eval(Image.open(p).convert("L")) for p in sub.path])
with torch.no_grad():
    p_nb = torch.softmax(model.eval()(xs.to(DEVICE)).cpu() / TEMPERATURE, 1)[:, 1].numpy()
print("max |module - notebook| probability difference:", float(np.abs(p_mod - p_nb).max()))
assert np.allclose(p_mod, p_nb, atol=1e-4), "deployment module does not reproduce notebook predictions"
print("✓ weights + config reload reproduce the notebook's predictions")

## 10. Demo inference — what the backend gets
`predict` (one tile, with test-time augmentation), `tile_mask` (Grad-CAM mask), `analyze` (tiles a **large** scene, mask + geometry).
`pixel_size_m` is an **assumption** (Sentinel-1 GRD ≈ 10 m) — set it to the real ground sampling distance of the scene.

In [ ]:
pos = tr[(tr.label == 1)].sort_values("p", ascending=False).head(1).iloc[0]
neg = tr[(tr.label == 0)].sort_values("p").head(1).iloc[0]
fig, axs = plt.subplots(2, 3, figsize=(10, 6.4))
EX = {}
for row, (name, r) in enumerate([("oil-like tile", pos), ("clean-sea tile", neg)]):
    res = pred.predict(r.path)
    mask, cam = pred.tile_mask(r.path)
    gray = np.asarray(Image.open(r.path).convert("L"))
    ov = C.OilSpillPredictor.render_overlay(gray, mask)
    ov.save(os.path.join(OUT_DIR, f"example_overlay_{'oil' if row == 0 else 'clean'}.png"))
    geo = C._geometry(mask, 10.0)
    EX[name] = {"result": res, "geometry": geo, "true_label": int(r.label)}
    axs[row, 0].imshow(gray, cmap="gray"); axs[row, 0].set_title(f"{name}  (true: {C.CLASS_NAMES[int(r.label)]})", fontsize=8)
    axs[row, 1].imshow(cam, cmap="inferno"); axs[row, 1].set_title("Grad-CAM", fontsize=8)
    axs[row, 2].imshow(ov); axs[row, 2].set_title(f"{res['prediction']}  P(oil)={res['oil_probability']:.2f}  conf={res['confidence']}", fontsize=8)
    for a in axs[row]: a.axis("off")
plt.tight_layout(); plt.show()
print(json.dumps(EX["oil-like tile"], indent=2)[:1400])

In [ ]:
# Large-scene path: a 3x3 mosaic of test tiles (a functionality test of tiling, NOT a real scene) -> 1200 x 1200 px
rng = np.random.default_rng(3)
pick = pd.concat([tr[tr.label == 1].sample(min(3, int((tr.label == 1).sum())), random_state=1),
                  tr[tr.label == 0].sample(min(6, int((tr.label == 0).sum())), random_state=1)]).sample(frac=1, random_state=2)
pick = pick.head(9)
mosaic = np.zeros((1200, 1200), np.uint8)
for k, (_, r) in enumerate(pick.iterrows()):
    im = Image.open(r.path).convert("L").resize((400, 400)); i, j = divmod(k, 3)
    mosaic[i * 400:(i + 1) * 400, j * 400:(j + 1) * 400] = np.asarray(im)
out = pred.analyze(mosaic, pixel_size_m=10.0)
R = out["result"]
print(f"scene {R['image_size_px']} px | tiles {R['n_tiles']} | positive tiles {R['n_positive_tiles']} | "
      f"scene P(oil) {R['oil_probability']} -> {R['prediction']} ({R['confidence']})")
print("geometry (assumed 10 m pixels):", R["geometry"])
ov = C.OilSpillPredictor.render_overlay(out["gray"], out["mask"]); ov.save(os.path.join(OUT_DIR, "example_scene_overlay.png"))
fig, ax = plt.subplots(1, 3, figsize=(13, 4.4))
ax[0].imshow(out["gray"], cmap="gray"); ax[0].set_title("scene (mosaic test)"); ax[1].imshow(out["prob_map"], cmap="magma", vmin=0, vmax=1); ax[1].set_title("tile-probability map")
ax[2].imshow(ov); ax[2].set_title("approximate mask overlay")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()

contract = {"system": "OILTRACE AI", "module": "CNN", "model_config": {k: config[k] for k in ("image_size", "decision_threshold", "temperature", "class_map")},
            "evaluation": {k: test_metrics[k] for k in ("accuracy", "precision", "recall", "roc_auc", "ci95_scene_bootstrap", "split", "n_test")},
            "tile_example": EX, "scene_example": {k: v for k, v in R.items()}, "disclaimer": C.DISCLAIMER}
json.dump(contract, open(os.path.join(OUT_DIR, "cnn_output_example.json"), "w"), indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
print("wrote cnn_output_example.json")

## 11. Automated tests
If this prints `ALL TESTS PASSED`, the CNN deliverable behaves as documented.

In [ ]:
tests = []
def check(name, cond):
    tests.append((name, bool(cond))); print(("PASS  " if cond else "FAIL  ") + name)

check("no scene shared between train/val/test", not (S["train"] & S["test"]) and not (S["train"] & S["val"]) and not (S["val"] & S["test"]))
check("no duplicate image bytes across the dataset", df.md5.is_unique)
check("all three splits contain both classes", all(df[df.split == k].label.nunique() == 2 for k in ("train", "val", "test")))
check("threshold chosen inside notebook and in (0,1)", 0 < THRESHOLD < 1)
check("temperature positive", TEMPERATURE > 0)
check("probabilities within [0,1]", ((pt >= 0) & (pt <= 1)).all())
check("deployment module reproduces notebook probabilities", np.allclose(p_mod, p_nb, atol=1e-4))
need = {"oil_probability", "prediction", "class", "confidence", "tta_std", "decision_threshold", "lookalike_analysis"}
check("tile result has the documented keys", need <= set(pred.predict(pos.path)))
check("look-alike breakdown is honestly marked unavailable", pred.predict(pos.path)["lookalike_analysis"]["breakdown_available"] is False)
check("prediction label matches threshold rule", all((r["class"] == int(r["oil_probability"] >= THRESHOLD)) for r in [pred.predict(pos.path), pred.predict(neg.path)]))
check("blank image is flagged as unreliable", len(pred.predict(np.zeros((400, 400), np.uint8))["warnings"]) == 1)
check("float raster with NaN is accepted without error", 0 <= pred.predict(np.where(np.random.rand(400, 400) < .02, np.nan, np.random.rand(400, 400)))["oil_probability"] <= 1)
check("scene confidence uses best-tile uncertainty (no tile-spread penalty)", R["tta_std"] == R["best_tile"]["tta_std"])
check("TTA std is non-negative", pred.predict(pos.path)["tta_std"] >= 0)
check("mask has the same shape as the scene", out["mask"].shape == out["gray"].shape)
check("scene tiling covers the mosaic (25 tiles)", R["n_tiles"] == 25)
check("geometry non-negative", R["geometry"]["area_km2"] >= 0)
check("png-bytes / numpy / PIL inputs all accepted", all(0 <= pred.predict(x, tta=False)["oil_probability"] <= 1 for x in
      [open(pos.path, "rb").read(), np.asarray(Image.open(pos.path)), Image.open(pos.path)]))
check("config json has required keys", {"image_size", "normalize_mean", "normalize_std", "decision_threshold", "temperature", "class_map", "weights_file"} <= set(json.load(open(os.path.join(OUT_DIR, "cnn_config.json")))))
check("weights file is a plain state_dict", all(isinstance(v, torch.Tensor) for v in torch.load(os.path.join(OUT_DIR, WEIGHTS_FILE), map_location="cpu").values()))
check("output contract round-trips", json.load(open(os.path.join(OUT_DIR, "cnn_output_example.json")))["module"] == "CNN")
if not SMOKE_TEST:
    check("test ROC-AUC above 0.85 (sanity floor)", M["roc_auc"] > 0.85)
check("reduced-size runs are flagged inside the config", config["evaluation"]["smoke_test"] == SMOKE_TEST)
bad = [n for n, ok in tests if not ok]
print(); print("ALL TESTS PASSED" if not bad else f"FAILED: {bad}"); assert not bad

## 12. Handoff package

In [ ]:
req = f"torch=={torch.__version__.split('+')[0]}\ntorchvision=={torchvision.__version__.split('+')[0]}\npillow\nnumpy\nscipy\n"
open(os.path.join(OUT_DIR, "requirements_cnn.txt"), "w").write(req)
files = [os.path.join(OUT_DIR, f) for f in sorted(os.listdir(OUT_DIR))] + ["oiltrace_cnn.py"]
with zipfile.ZipFile("OILTRACE_CNN_HANDOFF.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in files:
        z.write(f, arcname=os.path.basename(f)); print("✓", f)
print("\nCreated OILTRACE_CNN_HANDOFF.zip")
try:
    from google.colab import files as colab_files
    colab_files.download("OILTRACE_CNN_HANDOFF.zip")
except Exception:
    print("(Not in Colab - take the zip from the working directory.)")

## 13. Backend integration

```python
# backend/services/spill_detection.py
from oiltrace_cnn import OilSpillPredictor
PRED = OilSpillPredictor.load("models/cnn")            # once at startup; reads cnn_config.json + weights

def detect(image_bytes: bytes, pixel_size_m: float = 10.0):
    out = PRED.analyze(image_bytes, pixel_size_m=pixel_size_m)     # tile or large scene
    r = out["result"]
    return {**r,
            "mask_png_b64":    PRED.to_base64_png(out["mask"]),
            "overlay_png_b64": PRED.to_base64_png(PRED.render_overlay(out["gray"], out["mask"]))}
```

| Endpoint | Maps to |
|---|---|
| `POST /detect-spill` | `PRED.analyze(...)` → `oil_probability`, `prediction`, `confidence`, `n_positive_tiles` |
| `POST /characterize-spill` | `result["geometry"]` → `area_km2`, `length_km`, `width_km`, `orientation_deg` |
| Look-alike panel | **not provided by this model** (`breakdown_available: false`) – the demo engine may fill it, labelled DEMO |
| Demo mode | serve `cnn_output_example.json` + overlay PNGs; no GPU needed |

### Limitations to state honestly
- Binary patch classifier; the **mask and geometry are approximate** (Grad-CAM + darkness), not a trained segmentation.
- `pixel_size_m` is an input assumption; area/length/width scale with it.
- Trained on a single public dataset; performance on other sensors, polarisations, regions or sea states is unknown.
- Label meaning (`folder 1 = oil`) comes from the dataset's naming; confirm with the data owner.
- Report the **grouped-split test metrics with their confidence interval** — not numbers from a random split.